# E08 03 - Router con LangGraph + Langfuse (Starter)

Convertimos el router manual en un grafo:

```txt
START -> classify_node -> add_conditional_edges -> agente -> END
```

La funcion `route_to_node` no ejecuta agentes. Solo devuelve el nombre del nodo destino.


## Antes de tocar codigo: que estamos construyendo

Este notebook esta pensado para que puedas entenderlo aunque lo abras sin ver la clase.

Tema: **Router condicional con LangGraph + Langfuse**.

La regla didactica es:

1. primero explicamos el concepto;
2. despues mostramos el codigo minimo;
3. despues conectamos ese codigo con el paso anterior;
4. finalmente ejecutamos y leemos el resultado.

Cuando veas una funcion, preguntate:

- que recibe;
- que devuelve;
- que parte del flujo representa;
- si es logica de negocio, orquestacion o instrumentacion.


In [ ]:
# Esta celda instala las dependencias del notebook.
# En Google Colab cada notebook arranca con un entorno limpio, por eso instalamos al inicio.
# En local, si ya instalaste estos paquetes, pip simplemente confirmara que existen.
!pip install -q langfuse langchain langchain-openai langgraph

print('Dependencias instaladas: langfuse langchain langchain-openai langgraph')


In [ ]:
import os
from getpass import getpass

# Langfuse usa dos credenciales del proyecto: public key y secret key.
# Se obtienen en Langfuse Cloud, dentro de Settings -> API Keys.
os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key: ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key: ')

# URL del servicio. Para la region US se puede cambiar por https://us.cloud.langfuse.com.
os.environ['LANGFUSE_BASE_URL'] = 'https://cloud.langfuse.com'

# OpenAI sigue siendo necesario porque el grafo llama al LLM.
os.environ['OPENAI_API_KEY'] = getpass('OpenAI API Key: ')

print('OpenAI y Langfuse configurados para esta sesion.')


## Imports y estado

El State guarda:

- `query`: entrada original;
- `intent`: escrito por el clasificador;
- `response`: escrito por el agente elegido.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langfuse.langchain import CallbackHandler

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

class RouterState(TypedDict):
    query: str
    intent: str
    response: str


## Base de conocimiento simple

Todavia no hacemos RAG. La base esta en memoria porque el foco de E08 es routing.

En un RAG real, estos textos vendrian de documentos, se partirian en chunks y se buscarian con embeddings.


In [ ]:
# Tres dominios de ejemplo.
# Cada dominio tiene respuestas cortas para que podamos ver que rama ejecuto el router.
knowledge_base = {
    'hr': [
        'Vacaciones: se solicitan en el portal de RR. HH. con aprobacion del manager.',
        'Licencias: deben registrarse con motivo y fechas estimadas.',
    ],
    'tech': [
        'VPN: reiniciar cliente, validar 2FA y abrir ticket si persiste.',
        'Contrasena: restablecer desde el portal de identidad corporativa.',
    ],
    'finance': [
        'Facturas: cargar comprobantes antes del cierre mensual.',
        'Reembolsos: adjuntar recibo, monto y centro de costo.',
    ],
}

def answer_from_domain(domain: str) -> str:
    # Une las respuestas del dominio elegido.
    return ' | '.join(knowledge_base[domain])


## Nodos

Cada nodo escribe una parte del estado.


In [ ]:
classification_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Clasifica consultas internas en: hr, tech, finance o unknown.'),
    ('human', 'Consulta: {query}\nResponde solo la categoria.'),
])

classifier_chain = classification_prompt | llm | StrOutputParser()

def classify_node(state: RouterState) -> dict:
    intent = classifier_chain.invoke({'query': state['query']}).strip().lower()
    if intent not in {'hr', 'tech', 'finance'}:
        intent = 'unknown'
    return {'intent': intent}

def hr_node(state: RouterState) -> dict:
    return {'response': 'HRAgent: ' + answer_from_domain('hr')}

def tech_node(state: RouterState) -> dict:
    return {'response': 'TechAgent: ' + answer_from_domain('tech')}

def finance_node(state: RouterState) -> dict:
    return {'response': 'FinanceAgent: ' + answer_from_domain('finance')}

def unknown_node(state: RouterState) -> dict:
    return {'response': 'No tengo informacion suficiente. Puedo ayudar con HR, Tech o Finance.'}

def route_to_node(state: RouterState) -> str:
    # TODO 1: leer state['intent'].
    # TODO 2: devolver hr_node, tech_node, finance_node o unknown_node.
    raise NotImplementedError('Completar route_to_node')


## Grafo

Aca se practica `add_conditional_edges`, la pieza central de E08.


In [ ]:
# TODO 3: crear StateGraph(RouterState).
# TODO 4: agregar nodos.
# TODO 5: conectar START -> classify_node.
# TODO 6: usar add_conditional_edges.
# TODO 7: conectar cada rama a END.
# TODO 8: compilar el grafo.
graph = None


In [ ]:
# TODO 9: crear CallbackHandler().
langfuse_handler = None

# TODO 10: invocar el grafo pasando callbacks en config.
